# Daily Scanner Candidates v0.3 - Run Inspection

Notebook de inspeccion para entender un run de `daily_scanner_candidates_table_v0_3`.

Este notebook es de inspeccion/research. No define semantica oficial, no materializa el dataset oficial y no convierte filas del scanner en `market_state`, labels, rewards, senales, fills o PnL.

Autoridad semantica:

- `01_TSIS_backtest_SmallCaps/01_foundations/module_contracts/outputs/daily_scanner_candidates_table_target_contract_v0_3.md`
- `01_TSIS_backtest_SmallCaps/01_foundations/module_contracts/outputs/scanner_framework_and_definitions_contract_v0_3.md`
- `01_TSIS_backtest_SmallCaps/01_foundations/data_consumption_policies/daily_scanner_candidates_table_consumption_policy.md`
- `01_TSIS_backtest_SmallCaps/01_foundations/validators/outputs/daily_scanner_candidates_table_validators.md`

## Lectura correcta

`base_eligible_smallcap_denominator_v0_3` responde: que smallcaps puede mirar TSIS.

`selected_in_play_momentum_candidate` responde: que smallcaps pasaron movimiento fuerte y tradability.

`trade_station_like_profile_v0_3` responde: que habria visto un perfil operativo tipo TradeStation.

DAS y cualquier estrategia futura deben aplicar overlays despues. No forman parte del scanner global.

In [ ]:
from pathlib import Path
import json
import math
import textwrap

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    plt.style.use("default")

pd.set_option("display.max_columns", 220)
pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 180)

REPO_ROOT = Path(r"C:/TSIS_Data")
MODULE_ROOT = REPO_ROOT / "01_TSIS_backtest_SmallCaps"

DEFAULT_RUN_ROOT = REPO_ROOT / "tests" / "test_runs" / "2026-06-30" / "daily_scanner_candidates_v0_3_runner_smoke_20250102_20250110_d"
FALLBACK_DIRECT_REPLAY_ROOT = REPO_ROOT / "tests" / "test_runs" / "2026-06-30" / "daily_scanner_candidates_replay_20250102_20250110_v0_3_0_in_play_momentum"

# Cambia esta ruta para inspeccionar otro run v0.3.
RUN_ROOT = DEFAULT_RUN_ROOT if DEFAULT_RUN_ROOT.exists() else FALLBACK_DIRECT_REPLAY_ROOT

# Para runs de 20 anos, el notebook no debe cargar todo en pandas. DuckDB hace los agregados y pandas solo toma muestras.
ROW_LIMIT_FOR_PANDAS = 300_000

print("RUN_ROOT:", RUN_ROOT)
print("exists:", RUN_ROOT.exists())

## Descubrimiento de artefactos

Soporta dos layouts:

1. Runner por partes anuales: `parts/year=YYYY/daily_scanner_candidates_table_v0_3_candidate_replay/data.parquet`.
2. Replay directo: `daily_scanner_candidates_table_v0_3_candidate_replay/data.parquet`.

In [ ]:
def read_json(path: Path):
    if not path.exists():
        return None
    try:
        text = path.read_text(encoding="utf-8-sig")
        if not text.strip():
            return None
        return json.loads(text)
    except json.JSONDecodeError as exc:
        return {"_json_parse_error": str(exc), "_path": str(path)}

def discover_v0_3_parquets(run_root: Path):
    part_parquets = sorted(run_root.glob("parts/year=*/daily_scanner_candidates_table_v0_3_candidate_replay/data.parquet"))
    if part_parquets:
        return part_parquets, "runner_year_parts"
    direct = run_root / "daily_scanner_candidates_table_v0_3_candidate_replay" / "data.parquet"
    if direct.exists():
        return [direct], "direct_replay"
    raise FileNotFoundError(f"No v0.3 parquet found under {run_root}")

def discover_manifests(run_root: Path):
    return {
        "run_summary": sorted(run_root.glob("_run_summary.json")),
        "pre_manifest": sorted(run_root.glob("*.pre_manifest.json")),
        "heartbeat": sorted(run_root.glob("*.heartbeat.json")),
        "pids": sorted(run_root.glob("*.pids.json")),
        "part_manifests": sorted(run_root.glob("parts/year=*/_daily_scanner_candidates_table_manifest_v0_3_candidate_replay.json")),
        "direct_manifest": sorted(run_root.glob("_daily_scanner_candidates_table_manifest_v0_3_candidate_replay.json")),
        "part_summaries": sorted(run_root.glob("parts/year=*/_daily_scanner_candidates_table_summary_v0_3_candidate_replay.csv")),
        "direct_summary": sorted(run_root.glob("_daily_scanner_candidates_table_summary_v0_3_candidate_replay.csv")),
    }

parquet_files, layout = discover_v0_3_parquets(RUN_ROOT)
manifest_paths = discover_manifests(RUN_ROOT)

print("layout:", layout)
print("parquet files:", len(parquet_files))
for p in parquet_files[:10]:
    print(" -", p)
if len(parquet_files) > 10:
    print("...")

artifact_table = []
for k, paths in manifest_paths.items():
    artifact_table.append({"artifact_type": k, "count": len(paths), "first_path": str(paths[0]) if paths else None})
display(pd.DataFrame(artifact_table))

## Manifests y metadata del run

Esta celda muestra que se ejecuto, con que scope y que nivel de promocion tiene. Si `full_universe_claim=false`, no debe tratarse como dataset institucional completo.

In [ ]:
run_summary = read_json(manifest_paths["run_summary"][0]) if manifest_paths["run_summary"] else None
pre_manifest = read_json(manifest_paths["pre_manifest"][0]) if manifest_paths["pre_manifest"] else None
heartbeat = read_json(manifest_paths["heartbeat"][0]) if manifest_paths["heartbeat"] else None
pids = read_json(manifest_paths["pids"][0]) if manifest_paths["pids"] else None

def flatten_top_level(obj, prefix):
    if not obj:
        return pd.DataFrame(columns=["source", "key", "value"])
    rows = []
    for k, v in obj.items():
        if isinstance(v, (dict, list)):
            value = json.dumps(v, ensure_ascii=False)[:500]
        else:
            value = v
        rows.append({"source": prefix, "key": k, "value": value})
    return pd.DataFrame(rows)

display(pd.concat([
    flatten_top_level(run_summary, "run_summary"),
    flatten_top_level(pre_manifest, "pre_manifest"),
    flatten_top_level(heartbeat, "heartbeat"),
], ignore_index=True))

part_manifest_records = []
for path in (manifest_paths["part_manifests"] or manifest_paths["direct_manifest"]):
    m = read_json(path)
    part_manifest_records.append({
        "manifest": str(path),
        "dataset_id": m.get("dataset_id"),
        "promotion_level": m.get("promotion_level"),
        "status": m.get("status"),
        "scanner_run_id": m.get("scanner_run_id"),
        "start_date": (m.get("replay_scope") or {}).get("start_date"),
        "end_date": (m.get("replay_scope") or {}).get("end_date"),
        "full_universe_claim": (m.get("replay_scope") or {}).get("full_universe_claim"),
        "tree_sha256": (m.get("output_tree") or {}).get("tree_sha256"),
    })
display(pd.DataFrame(part_manifest_records))

## Cargar tabla con DuckDB

DuckDB permite analizar runs grandes sin meter todo en memoria. Para graficos se carga una muestra deterministica si el run supera `ROW_LIMIT_FOR_PANDAS`.

In [ ]:
def sql_quote_path(path: Path) -> str:
    return "'" + path.as_posix().replace("'", "''") + "'"

parquet_list_sql = "[" + ",".join(sql_quote_path(p) for p in parquet_files) + "]"
SCAN = f"read_parquet({parquet_list_sql}, union_by_name=true)"
con = duckdb.connect()

row_count = con.sql(f"select count(*) from {SCAN}").fetchone()[0]
columns_df = con.sql(f"describe select * from {SCAN}").fetchdf()
print("row_count:", row_count)
display(columns_df)

if row_count <= ROW_LIMIT_FOR_PANDAS:
    df_plot = con.sql(f"select * from {SCAN}").fetchdf()
    sample_note = "full table loaded into pandas"
else:
    df_plot = con.sql(f"select * from {SCAN} order by hash(ticker, session_date, instrument_id) limit {ROW_LIMIT_FOR_PANDAS}").fetchdf()
    sample_note = f"deterministic sample loaded into pandas: {ROW_LIMIT_FOR_PANDAS} rows"
print(sample_note)
display(df_plot.head(20))

## KPIs principales

Estos KPIs separan poblacion evaluada, base elegible, in-play momentum y perfil TradeStation-like.

In [ ]:
kpi = con.sql(f"""
select
    count(*) as rows,
    count(distinct session_date) as sessions,
    count(distinct instrument_id) as instruments,
    count(distinct ticker) as tickers,
    sum(case when all_filters_passed then 1 else 0 end) as base_eligible_rows,
    sum(case when selected_in_play_momentum_candidate then 1 else 0 end) as in_play_rows,
    sum(case when selected_trade_station_like_profile then 1 else 0 end) as trade_station_like_rows,
    sum(case when selected_das_research_profile then 1 else 0 end) as das_rows,
    sum(case when selected_in_play_momentum_candidate and not in_play_motion_threshold_passed then 1 else 0 end) as selected_without_50_move,
    sum(case when selected_in_play_momentum_candidate and not in_play_volume_tradability_passed then 1 else 0 end) as selected_without_tradability,
    min(case when selected_in_play_momentum_candidate then in_play_motion_pct end) as min_selected_motion_pct,
    max(case when selected_in_play_momentum_candidate then in_play_motion_pct end) as max_selected_motion_pct
from {SCAN}
""").fetchdf()

display(kpi.T.rename(columns={0: "value"}))

In [ ]:
k = kpi.iloc[0].to_dict()
denom = pd.DataFrame([
    {"stage": "evaluated rows", "rows": int(k["rows"])},
    {"stage": "base eligible", "rows": int(k["base_eligible_rows"])},
    {"stage": "in-play momentum", "rows": int(k["in_play_rows"])},
    {"stage": "TradeStation-like", "rows": int(k["trade_station_like_rows"])},
])

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(denom["stage"], denom["rows"], color=["#6c757d", "#2b8a3e", "#d9480f", "#1c7ed6"])
ax.set_title("Scanner v0.3 - denominador y seleccion")
ax.set_ylabel("Filas")
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f"{int(bar.get_height()):,}", ha="center", va="bottom")
plt.xticks(rotation=20, ha="right")
plt.show()

display(denom)

## Evolucion diaria

Sirve para ver si el scanner produce masas razonables por dia y si las fechas tienen comportamientos extremos.

In [ ]:
daily = con.sql(f"""
select
    cast(session_date as date) as session_date,
    count(*) as evaluated_rows,
    sum(case when all_filters_passed then 1 else 0 end) as base_eligible_rows,
    sum(case when selected_in_play_momentum_candidate then 1 else 0 end) as in_play_rows,
    sum(case when selected_trade_station_like_profile then 1 else 0 end) as trade_station_like_rows,
    sum(case when selected_in_play_momentum_candidate and selected_trade_station_like_profile then 1 else 0 end) as both_rows
from {SCAN}
group by 1
order by 1
""").fetchdf()
display(daily)

fig, ax = plt.subplots(figsize=(13, 6))
for col, color in [
    ("base_eligible_rows", "#2b8a3e"),
    ("in_play_rows", "#d9480f"),
    ("trade_station_like_rows", "#1c7ed6"),
    ("both_rows", "#7048e8"),
]:
    ax.plot(daily["session_date"], daily[col], marker="o", label=col, color=color)
ax.set_title("Filas por dia: base eligible vs in-play vs TradeStation-like")
ax.set_ylabel("N filas")
ax.legend()
plt.xticks(rotation=30, ha="right")
plt.show()

## Solape entre in-play momentum y TradeStation-like

Este grafico responde si el perfil operativo humano captura los mismos tickers que el denominador in-play momentum, o si hay candidatos que uno ve y el otro no.

In [ ]:
overlap = con.sql(f"""
select
    case
        when selected_in_play_momentum_candidate and selected_trade_station_like_profile then 'both'
        when selected_in_play_momentum_candidate and not selected_trade_station_like_profile then 'in_play_only'
        when not selected_in_play_momentum_candidate and selected_trade_station_like_profile then 'tradestation_only'
        when all_filters_passed then 'base_only'
        else 'not_base'
    end as bucket,
    count(*) as rows
from {SCAN}
group by 1
order by rows desc
""").fetchdf()
display(overlap)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(overlap["bucket"], overlap["rows"], color="#495057")
ax.set_title("Solape scanner: in-play momentum vs TradeStation-like")
ax.set_ylabel("Filas")
for i, r in overlap.iterrows():
    ax.text(i, r["rows"], f"{int(r['rows']):,}", ha="center", va="bottom")
plt.xticks(rotation=25, ha="right")
plt.show()

## Distribucion de movimiento

La pregunta aqui es si el umbral `50%` esta filtrando de forma coherente. `in_play_motion_pct` usa el mayor proxy diario disponible entre high vs previous close, close vs previous close y gap.

In [ ]:
motion_cols = ["pct_chg_1d", "gap_pct", "daily_high_vs_prev_close_pct", "in_play_motion_pct"]
plot_motion = df_plot[[c for c in motion_cols + ["all_filters_passed", "selected_in_play_momentum_candidate"] if c in df_plot.columns]].copy()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()
for ax, col in zip(axes, motion_cols):
    if col not in plot_motion:
        ax.set_visible(False)
        continue
    base_values = plot_motion.loc[plot_motion["all_filters_passed"].fillna(False), col].dropna()
    selected_values = plot_motion.loc[plot_motion["selected_in_play_momentum_candidate"].fillna(False), col].dropna()
    ax.hist(base_values, bins=60, alpha=0.55, label="base eligible", color="#adb5bd")
    ax.hist(selected_values, bins=40, alpha=0.75, label="selected in-play", color="#d9480f")
    ax.axvline(50, color="black", linestyle="--", linewidth=1, label="50% threshold")
    ax.set_title(col)
    ax.set_xlabel("%")
    ax.set_ylabel("frecuencia")
    ax.legend()
plt.tight_layout()
plt.show()

display(con.sql(f"""
select
    min(in_play_motion_pct) filter (where selected_in_play_momentum_candidate) as min_selected_motion,
    median(in_play_motion_pct) filter (where selected_in_play_momentum_candidate) as median_selected_motion,
    max(in_play_motion_pct) filter (where selected_in_play_momentum_candidate) as max_selected_motion,
    min(in_play_motion_pct) filter (where all_filters_passed and not selected_in_play_momentum_candidate) as min_base_not_selected_motion,
    median(in_play_motion_pct) filter (where all_filters_passed and not selected_in_play_momentum_candidate) as median_base_not_selected_motion,
    max(in_play_motion_pct) filter (where all_filters_passed and not selected_in_play_momentum_candidate) as max_base_not_selected_motion
from {SCAN}
""").fetchdf())

## Volumen y tradability

La tradability v0.3 es `volume_today >= 500k OR dollar_volume_today >= 250k`. Esta seccion muestra si los candidatos seleccionados tienen suficiente actividad negociada.

In [ ]:
vol_cols = ["volume_today", "dollar_volume_today", "in_play_motion_pct", "selected_in_play_momentum_candidate", "all_filters_passed"]
vol_df = df_plot[[c for c in vol_cols if c in df_plot.columns]].copy()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, col, threshold in [(axes[0], "volume_today", 500_000), (axes[1], "dollar_volume_today", 250_000)]:
    values = vol_df.loc[vol_df["all_filters_passed"].fillna(False), col].dropna()
    selected = vol_df.loc[vol_df["selected_in_play_momentum_candidate"].fillna(False), col].dropna()
    ax.hist(values, bins=70, alpha=0.5, label="base eligible", color="#adb5bd")
    ax.hist(selected, bins=50, alpha=0.75, label="selected in-play", color="#d9480f")
    ax.axvline(threshold, color="black", linestyle="--", label=f"threshold {threshold:,}")
    ax.set_yscale("log")
    ax.set_title(col)
    ax.set_ylabel("frecuencia log")
    ax.legend()
plt.tight_layout()
plt.show()

scatter_df = vol_df.dropna(subset=["volume_today", "in_play_motion_pct"]).copy()
fig, ax = plt.subplots(figsize=(10, 6))
colors = np.where(scatter_df["selected_in_play_momentum_candidate"].fillna(False), "#d9480f", "#868e96")
ax.scatter(scatter_df["volume_today"].clip(lower=1), scatter_df["in_play_motion_pct"], s=18, alpha=0.55, c=colors)
ax.axhline(50, color="black", linestyle="--", linewidth=1)
ax.axvline(500_000, color="black", linestyle=":", linewidth=1)
ax.set_xscale("log")
ax.set_title("Movimiento vs volumen negociado")
ax.set_xlabel("volume_today log")
ax.set_ylabel("in_play_motion_pct")
plt.show()

## Precio, market cap y float

Market cap es filtro global `<100M`. Float no es filtro global todavia.

Importante: en v0.3 `float_shares` es una columna reservada por contrato, pero debe salir vacia mientras no exista una fuente point-in-time validada. No debe confundirse con `overview_weighted_shares_outstanding`, que son shares outstanding/diluted shares desde `instrument_master`, no float negociable.

In [ ]:
ctx_cols = ["last_price", "market_cap_usd", "float_shares", "in_play_motion_pct", "all_filters_passed", "selected_in_play_momentum_candidate"]
ctx = df_plot[[c for c in ctx_cols if c in df_plot.columns]].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, threshold in [
    (axes[0], "last_price", None),
    (axes[1], "market_cap_usd", 100_000_000),
    (axes[2], "float_shares", None),
]:
    if col not in ctx:
        ax.set_visible(False)
        continue
    values = ctx.loc[ctx["all_filters_passed"].fillna(False), col].dropna()
    selected = ctx.loc[ctx["selected_in_play_momentum_candidate"].fillna(False), col].dropna()
    ax.hist(values, bins=60, alpha=0.5, color="#adb5bd", label="base eligible")
    ax.hist(selected, bins=40, alpha=0.75, color="#d9480f", label="selected in-play")
    if threshold:
        ax.axvline(threshold, color="black", linestyle="--")
    if col in {"market_cap_usd", "float_shares"}:
        ax.set_xscale("log")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

display(con.sql(f"""
select
    float_filter_state,
    count(*) as rows,
    sum(case when selected_in_play_momentum_candidate then 1 else 0 end) as selected_rows,
    count(float_shares) as rows_with_float
from {SCAN}
group by 1
order by rows desc
""").fetchdf())

## Auditoria explicita de float

Esta celda responde si el run trae float real. Resultado esperado en v0.3: `float_shares` vacio y `float_filter_state = not_used_until_point_in_time_float_source_exists`.

Tambien revisa `instrument_master` para ver si hay columnas parecidas a shares. Si aparece `overview_weighted_shares_outstanding`, tratarlo como shares outstanding/diluted shares, no como float institucional.

In [ ]:
float_audit = con.sql(f"""
select
    count(*) as rows,
    count(float_shares) as non_null_float_shares,
    count(float_asof_date) as non_null_float_asof_date,
    count(float_source) as non_null_float_source,
    sum(case when selected_in_play_momentum_candidate and float_shares is not null then 1 else 0 end) as selected_rows_with_float,
    min(float_filter_state) as min_float_filter_state,
    max(float_filter_state) as max_float_filter_state
from {SCAN}
""").fetchdf()
display(float_audit.T.rename(columns={0: "value"}))

display(con.sql(f"""
select float_filter_state, count(*) as rows
from {SCAN}
group by 1
order by rows desc
""").fetchdf())

instrument_master_path = Path(r"E:/TSIS/data/data_foundation_outputs/instrument_master/instrument_master_v0_1.parquet")
if instrument_master_path.exists():
    im_cols = con.sql(f"describe select * from read_parquet('{instrument_master_path.as_posix()}')").fetchdf()
    share_like_cols = im_cols.loc[im_cols['column_name'].str.contains('float|share|outstanding', case=False, regex=True), 'column_name'].tolist()
    print('instrument_master share-like columns:', share_like_cols)
    if share_like_cols:
        non_null_exprs = []
        for c in share_like_cols:
            if c != 'share_class_figi':
                non_null_exprs.append(f"count({c}) as non_null_{c}")
        if non_null_exprs:
            display(con.sql(f"select {', '.join(non_null_exprs)} from read_parquet('{instrument_master_path.as_posix()}')").fetchdf().T.rename(columns={0: 'non_null_rows'}))
        sample_cols = ['ticker']
        for c in ['overview_weighted_shares_outstanding', 'overview_market_cap', 'lt1b_shares_source', 'lt1b_shares_observed_date', 'lt1b_shares_age_days']:
            if c in im_cols['column_name'].tolist():
                sample_cols.append(c)
        display(con.sql(f"select {', '.join(sample_cols)} from read_parquet('{instrument_master_path.as_posix()}') limit 20").fetchdf())
else:
    print('instrument_master not found:', instrument_master_path)

print('Interpretacion: float_shares vacio significa que TSIS aun no tiene float point-in-time validado para este scanner. No usar overview_weighted_shares_outstanding como sustituto de float sin contrato separado.')

## Reasons y componentes de inclusion

Cuenta que razones estan activando candidatos. En el replay diario proxy, las razones intradia/segmento deben permanecer falsas hasta tener builder intradia.

In [ ]:
reason_cols = [c for c in columns_df["column_name"].tolist() if c.startswith("reason_")]
reason_select = ",\n".join([f"sum(case when {c} then 1 else 0 end) as {c}" for c in reason_cols])
reason_counts = con.sql(f"select {reason_select} from {SCAN}").fetchdf().T.reset_index()
reason_counts.columns = ["reason", "rows"]
reason_counts = reason_counts.sort_values("rows", ascending=False)
display(reason_counts)

fig, ax = plt.subplots(figsize=(13, max(5, 0.35 * len(reason_counts))))
ax.barh(reason_counts["reason"], reason_counts["rows"], color="#f08c00")
ax.invert_yaxis()
ax.set_title("Reason flags activas")
ax.set_xlabel("Filas")
plt.show()

display(con.sql(f"""
select candidate_reasons, count(*) as rows
from {SCAN}
where candidate_reasons is not null and candidate_reasons <> ''
group by 1
order by rows desc, candidate_reasons
limit 50
""").fetchdf())

## Casos frontera

Estas tablas ayudan a auditar decisiones: seleccionados no vistos por TradeStation, TradeStation que no pasan in-play v0.3, movimientos fuertes sin tradability y tradables sin movimiento fuerte.

In [ ]:
case_cols = """
session_date, ticker, instrument_id, last_price, market_cap_usd,
pct_chg_1d, gap_pct, daily_high_vs_prev_close_pct, in_play_motion_pct,
volume_today, dollar_volume_today, selected_in_play_momentum_candidate,
selected_trade_station_like_profile, rank_pct_chg_1d, rank_composite_in_play,
candidate_reasons
"""

queries = {
    "in_play_not_tradestation": f"""
        select {case_cols}
        from {SCAN}
        where selected_in_play_momentum_candidate and not selected_trade_station_like_profile
        order by session_date, in_play_motion_pct desc nulls last
        limit 100
    """,
    "tradestation_not_in_play": f"""
        select {case_cols}
        from {SCAN}
        where selected_trade_station_like_profile and not selected_in_play_momentum_candidate
        order by session_date, pct_chg_1d desc nulls last
        limit 100
    """,
    "strong_move_no_tradability": f"""
        select {case_cols}
        from {SCAN}
        where all_filters_passed and in_play_motion_threshold_passed and not in_play_volume_tradability_passed
        order by in_play_motion_pct desc nulls last
        limit 100
    """,
    "tradable_no_50_move": f"""
        select {case_cols}
        from {SCAN}
        where all_filters_passed and in_play_volume_tradability_passed and not in_play_motion_threshold_passed
        order by dollar_volume_today desc nulls last
        limit 100
    """,
}

for name, query in queries.items():
    print("\n" + "="*120)
    print(name)
    display(con.sql(query).fetchdf())

## Top candidatos por dia

Lista legible para inspeccion humana. Ordena por `in_play_motion_pct`, no por cierre, porque el objetivo es capturar pump/frontside aunque cierre destruido.

In [ ]:
top_daily = con.sql(f"""
select *
from (
    select
        session_date,
        ticker,
        last_price,
        market_cap_usd,
        pct_chg_1d,
        gap_pct,
        daily_high_vs_prev_close_pct,
        in_play_motion_pct,
        volume_today,
        dollar_volume_today,
        selected_trade_station_like_profile,
        rank_pct_chg_1d,
        candidate_reasons,
        row_number() over (partition by session_date order by in_play_motion_pct desc nulls last, dollar_volume_today desc nulls last) as rn
    from {SCAN}
    where selected_in_play_momentum_candidate
)
where rn <= 25
order by session_date, rn
""").fetchdf()

display(top_daily)

## Calidad, flags y uso permitido

El scanner puede ser util como denominador, pero no como feature store ML/RL ni autoridad live. Esta celda comprueba flags de calidad y prohibiciones.

In [ ]:
quality_cols = [c for c in columns_df["column_name"].tolist() if any(s in c for s in ["quality", "valid_for", "future", "available", "replay", "state", "claim"])]
display(pd.DataFrame({"quality_or_gate_column": quality_cols}))

gate_exprs = []
for c in [
    "full_universe_claim",
    "contains_future_information_without_event_filter",
    "valid_for_ml_feature_candidate",
    "valid_for_rl_state_candidate",
    "valid_for_live_downstream_candidate",
    "manual_research_seed",
    "live_scanner_candidate",
    "historical_replay_candidate",
    "scanner_replayable",
]:
    if c in columns_df["column_name"].tolist():
        gate_exprs.append(f"sum(case when {c} then 1 else 0 end) as {c}")

gate_df = con.sql(f"select {', '.join(gate_exprs)} from {SCAN}").fetchdf().T.reset_index()
gate_df.columns = ["gate", "true_rows"]
display(gate_df)

null_exprs = []
for c in ["ticker", "instrument_id", "session_date", "market_cap_usd", "last_price", "previous_close", "volume_today", "dollar_volume_today", "in_play_motion_pct"]:
    null_exprs.append(f"sum(case when {c} is null then 1 else 0 end) as {c}_nulls")
display(con.sql(f"select {', '.join(null_exprs)} from {SCAN}").fetchdf().T.rename(columns={0: "null_rows"}))

## Tabla resumen por ticker

Ayuda a detectar tickers que aparecen varias veces, tickers con muchos dias in-play o tickers dominantes en la muestra.

In [ ]:
ticker_summary = con.sql(f"""
select
    ticker,
    count(*) as rows,
    count(distinct session_date) as sessions,
    sum(case when all_filters_passed then 1 else 0 end) as base_rows,
    sum(case when selected_in_play_momentum_candidate then 1 else 0 end) as in_play_rows,
    sum(case when selected_trade_station_like_profile then 1 else 0 end) as tradestation_rows,
    max(in_play_motion_pct) as max_motion_pct,
    max(dollar_volume_today) as max_dollar_volume
from {SCAN}
group by 1
order by in_play_rows desc, max_motion_pct desc nulls last, max_dollar_volume desc nulls last
limit 100
""").fetchdf()
display(ticker_summary)

fig, ax = plt.subplots(figsize=(13, 6))
plot_ts = ticker_summary.head(30).copy()
ax.bar(plot_ts["ticker"], plot_ts["in_play_rows"], color="#d9480f")
ax.set_title("Top tickers por numero de sesiones in-play")
ax.set_ylabel("Sesiones in-play")
plt.xticks(rotation=60, ha="right")
plt.show()

## Interpretacion automatica minima

No sustituye al criterio humano. Solo resume checks que deberian quedar claros antes de usar el run como denominador de research.

In [ ]:
run_summary = globals().get("run_summary", None)
heartbeat = globals().get("heartbeat", None)

checks = []
k0 = kpi.iloc[0]
checks.append({"check": "selected rows require >=50% movement", "pass": int(k0["selected_without_50_move"]) == 0, "value": int(k0["selected_without_50_move"])})
checks.append({"check": "selected rows require tradability", "pass": int(k0["selected_without_tradability"]) == 0, "value": int(k0["selected_without_tradability"])})
checks.append({"check": "DAS is not inside global scanner", "pass": int(k0["das_rows"]) == 0, "value": int(k0["das_rows"])})
checks.append({"check": "has in-play candidates", "pass": int(k0["in_play_rows"]) > 0, "value": int(k0["in_play_rows"])})
checks.append({"check": "has base eligible denominator", "pass": int(k0["base_eligible_rows"]) > 0, "value": int(k0["base_eligible_rows"])})

if run_summary:
    checks.append({"check": "runner full_universe_claim is false", "pass": run_summary.get("full_universe_claim") is False, "value": run_summary.get("full_universe_claim")})
if heartbeat:
    checks.append({"check": "runner heartbeat completed", "pass": heartbeat.get("status") == "completed", "value": heartbeat.get("status")})

checks_df = pd.DataFrame(checks)
display(checks_df)

failed = checks_df.loc[~checks_df["pass"]]
if len(failed):
    print("ATTENTION: checks failed")
    display(failed)
else:
    print("All notebook-level sanity checks passed for declared scope.")

## Siguiente analisis recomendado

1. Ejecutar el runner v0.3 para un rango mayor con monitor compacto.
2. Comparar estabilidad de conteos por ano y regimen.
3. Construir el builder intradia de segmentos 04:00-20:00 para saber si el primer `50%` ocurrio en premarket, regular o after-hours.
4. Aplicar overlays de estrategia, por ejemplo DAS, sobre este denominador sin esconder casos negativos.
5. Solo despues componer `market_state_candidate` / `event_state_candidate` con as-of, ventanas, calidad y leakage gates.